In [23]:

from itertools import combinations
from decimal import Decimal, InvalidOperation

#cosmetic
def _dec_to_str(d: Decimal) -> str:
    s = format(d, "f")
    if "." in s:
        s = s.rstrip("0").rstrip(".")
    return s

def fmt_entry(x) -> str:
    return _dec_to_str(x) if isinstance(x, Decimal) else str(x)

def fmt_seq(seq) -> str:
    return " ".join(fmt_entry(x) for x in seq)

def parse_token(tok):
    """
    Parse token as:
      - int, if it is an integer literal (e.g. -3, 12)
      - Decimal, otherwise (e.g. 2.25, -1.5)
    """
    tok = str(tok).strip()
    if tok == "":
        raise ValueError("Empty token in Gauss code.")
    if tok.lstrip("+-").isdigit():
        return int(tok)
    try:
        return Decimal(tok)
    except InvalidOperation as e:
        raise ValueError(f"Could not parse token '{tok}'.") from e

def process_gauss_code(raw_gauss_code):
    """
      - if raw_gauss_code is a string: split on whitespace/commas
      - if it is a list/tuple: parse each entry
    Returns a list of (int or Decimal).
    """
    if isinstance(raw_gauss_code, str):
        s = raw_gauss_code.replace(",", " ").strip()
        parts = [p for p in s.split() if p.strip()]
        return [parse_token(p) for p in parts]
    return [parse_token(x) for x in raw_gauss_code]

def is_int_entry(x): return isinstance(x, int)
def is_negative_int(x): return isinstance(x, int) and x < 0
def is_nonint_entry(x): return isinstance(x, Decimal)
def abs_label(x): return -x if x < 0 else x


#CreateKnotDict

def is_breakpoint(x, has_nonints):
    """
    Original (integer-only): breakpoint = negative integer.
    With non-integers present: every non-integer (any sign) is ALSO a breakpoint.
    """
    return is_negative_int(x) or (has_nonints and is_nonint_entry(x))

def find_strands(gauss_code):
    """
    Integer-only (original behavior):
      strands are cyclic subsequences from negative int to next negative int (inclusive).

    With non-integers present:
      strands are cyclic subsequences from any breakpoint (neg int OR any non-int)
      to the next breakpoint (inclusive).
    """
    if not gauss_code:
        raise ValueError("Gauss code is empty.")

    has_nonints = any(is_nonint_entry(x) for x in gauss_code)
    bp = [i for i, x in enumerate(gauss_code) if is_breakpoint(x, has_nonints)]
    if not bp:
        raise ValueError("No breakpoints found. Need negative integers (and/or non-integers).")

    bp = sorted(bp)
    n = len(gauss_code)

    strand_list = []
    for j in range(len(bp)):
        a = bp[j]
        b = bp[(j + 1) % len(bp)]
        if a <= b:
            strand = tuple(gauss_code[a:b+1])
        else:
            strand = tuple(gauss_code[a:] + gauss_code[:b+1])
        strand_list.append(strand)

    # Deduplicate but keep deterministic order
    seen = set()
    strand_set = []
    for st in strand_list:
        if st not in seen:
            seen.add(st)
            strand_set.append(st)

    # Label A, B, C,... for strands
    letter_list = list(map(chr, range(65, 91)))
    strands_dict = {}
    for i, strand in enumerate(strand_set):
        lab = letter_list[i] if i < len(letter_list) else f"A{i}"
        strands_dict[lab] = [strand, []]
    return strands_dict

def find_crossings(knot_dict, gauss_code):
    """
    ORIGINAL integer-crossing logic (unchanged):
      For each positive integer 'under' appearing in a strand,
      find under1 = strand whose subseq starts with -under
           under2 = strand whose subseq ends   with -under
      and record (under1, under2) as "crossings over" for the over-strand.
    """
    for key_outer in knot_dict:
        for under in knot_dict[key_outer][0]:
            if is_int_entry(under) and under > 0:
                found1, found2 = False, False
                for key_inner in knot_dict:
                    if found1 and found2:
                        break
                    if knot_dict[key_inner][0][0] == -under:
                        under1 = key_inner
                        found1 = True
                    if knot_dict[key_inner][0][-1] == -under:
                        under2 = key_inner
                        found2 = True
                knot_dict[key_outer][1].append((under1, under2))
    return knot_dict

def build_nonint_junctions(knot_dict):
    """
    For each non-integer absolute label a, locate:
      end(+a), start(+a), end(-a), start(-a)
    Returns dict: a -> (end_plus, start_plus, end_minus, start_minus)
    """
    nonint_abs = set()
    for s in knot_dict:
        first = knot_dict[s][0][0]
        last  = knot_dict[s][0][-1]
        if is_nonint_entry(first): nonint_abs.add(abs_label(first))
        if is_nonint_entry(last):  nonint_abs.add(abs_label(last))

    junctions = {}
    for a in nonint_abs:
        end_plus = start_plus = end_minus = start_minus = None
        for s in knot_dict:
            first = knot_dict[s][0][0]
            last  = knot_dict[s][0][-1]
            if first == a:   start_plus = s
            if last == a:    end_plus = s
            if first == -a:  start_minus = s
            if last == -a:   end_minus = s

        if None in (end_plus, start_plus, end_minus, start_minus):
            raise ValueError(
                f"Non-integer label {fmt_entry(a)} did not determine 4 incident strands.\n"
                f"Found: end(+a)={end_plus}, start(+a)={start_plus}, end(-a)={end_minus}, start(-a)={start_minus}"
            )
        junctions[a] = (end_plus, start_plus, end_minus, start_minus)

    return junctions

def create_knot_dictionary(gauss_code):
    """
    Original returned just knot_dict.
    Here we also return 'junctions' for non-integer propagation.
    """
    strands_dict = find_strands(gauss_code)
    knot_dict = find_crossings(strands_dict, gauss_code)
    junctions = build_nonint_junctions(knot_dict)
    return knot_dict, junctions

#BringingInOrdering

def build_strand_rank(knot_dict, gauss_code):
    """
    Rank strands by the position of their starting entry (left->right)
    in the *current* gauss_code order.
    """
    start_entry = {s: knot_dict[s][0][0] for s in knot_dict}
    pos = {s: gauss_code.index(start_entry[s]) for s in knot_dict}
    order = sorted(knot_dict.keys(), key=lambda s: pos[s])
    return {s: i for i, s in enumerate(order)}

def is_valid_coloring(seed_strands, knot_dict, junctions, gauss_code):
    """
    ORDERED propagation for ONE fixed Gauss reading (this replaces the unordered rule).

    Integer crossings:
      if crossing = (u1, u2), allow u2 -> u1 only if rank[u1] > rank[u2].

    Non-integer junctions:
      for each non-integer label a:
        if end(+a) and end(-a) are colored, then color start(+a) and start(-a).
    """
    rank = build_strand_rank(knot_dict, gauss_code)

    seed_strands = set(seed_strands)
    colored_set = seed_strands.copy()
    new_coloring = True

    while new_coloring:
        new_coloring = False

        # integer crossings (directed)
        for strand in colored_set.copy():
            for (u1, u2) in knot_dict[strand][1]:
                if (u2 in colored_set) and (u1 not in colored_set) and (rank[u1] > rank[u2]):
                    colored_set.add(u1)
                    new_coloring = True

        # non-integer junctions
        for a, (end_p, start_p, end_m, start_m) in junctions.items():
            if (end_p in colored_set) and (end_m in colored_set):
                if start_p not in colored_set:
                    colored_set.add(start_p); new_coloring = True
                if start_m not in colored_set:
                    colored_set.add(start_m); new_coloring = True

    return colored_set == set(knot_dict.keys())

def calc_wirt_info(knot_dict, junctions, gauss_code):
    """
      Exhaustively search seed size n=1,2,3,... and return the first that works.
    """
    n = 2
    while n <= len(knot_dict):
        for seed_strands in combinations(knot_dict, n):
            if is_valid_coloring(seed_strands, knot_dict, junctions, gauss_code):
                return (seed_strands, n)
        n += 1

    # fallback
    all_strands = tuple(knot_dict.keys())
    return (all_strands, len(all_strands))

def calc_wirt_info_bidir(gauss_code):
    """
    Bidirectional definition:
      - compute ordered Wirtinger on the given order (forward)
      - compute ordered Wirtinger on reversed Gauss list (reverse)
      - return the smaller (tie-break: forward)
    Returns: (seed, n, direction, knot_dict, junctions)
    """
    # forward
    knot_dict_f, junctions_f = create_knot_dictionary(gauss_code)
    seed_f, n_f = calc_wirt_info(knot_dict_f, junctions_f, gauss_code)

    # reverse
    gauss_rev = list(reversed(gauss_code))
    knot_dict_r, junctions_r = create_knot_dictionary(gauss_rev)
    seed_r, n_r = calc_wirt_info(knot_dict_r, junctions_r, gauss_rev)

    if n_f <= n_r:
        return seed_f, n_f, "forward", knot_dict_f, junctions_f
    return seed_r, n_r, "reverse", knot_dict_r, junctions_r

#print

def print_knot_dictionary(knot_dict):
    print("\nKnot dictionary:\n")
    print("{:^15}{:45}{}".format("STRAND", "SUBSEQUENCE", "CROSSINGS OVER"))
    for strand in knot_dict:
        subsequence = knot_dict[strand][0]
        undercrossings = " ".join([str(x) for x in knot_dict[strand][1]])
        print("{:^15}{:45}{}".format(strand, f"({fmt_seq(subsequence)})", undercrossings))

def print_junctions(junctions):
    if not junctions:
        return
    print("\nNon-integer junctions (a -> (end(+a), start(+a), end(-a), start(-a))):")
    for a in sorted(junctions.keys(), key=lambda z: float(z)):
        print(f"  {fmt_entry(a)}: {junctions[a]}")

#running

def run_notebook(gauss_str=None, gauss_code=None, verbose=True, quiet=False, map_half=False):
    """
    Use either:
      run_notebook(gauss_str="...") OR run_notebook(gauss_code=[...])
    """
    if gauss_code is None:
        if gauss_str is None:
            raise ValueError("Provide gauss_str or gauss_code.")
        gauss_code = process_gauss_code(gauss_str)
    else:
        gauss_code = process_gauss_code(gauss_code)

    
    seed, wirt_num, direction, knot_dict, junctions = calc_wirt_info_bidir(gauss_code)

    if quiet:
        print(wirt_num)
        return wirt_num, seed, direction, knot_dict, junctions

    if verbose:
        print(f"\nGauss code used ({direction}): {fmt_seq(gauss_code if direction=='forward' else list(reversed(gauss_code)))}")
        print_knot_dictionary(knot_dict)
        print_junctions(junctions)
        print(f"\nSeed strand set: {seed}")
        print(f"Ordered Wirtinger number: {wirt_num}")
    else:
        print(f"Ordered Wirtinger number: {wirt_num}")

    


#Syntaxtorun

# Example 1: your earlier test (note: this one has decimals)
# gauss_str = "-1 -2.25 -2 3 -4 1 2.25 4 -3 2"
# run_notebook(gauss_str=gauss_str, verbose=True, quiet=False, map_half=False)

In [24]:
gauss_str = "-1 2 -3 1 -2 3"
run_notebook(gauss_str=gauss_str,verbose=True, quiet=False)


Gauss code used (forward): -1 2 -3 1 -2 3

Knot dictionary:

    STRAND     SUBSEQUENCE                                  CROSSINGS OVER
       A       (-1 2 -3)                                    ('C', 'B')
       B       (-3 1 -2)                                    ('A', 'C')
       C       (-2 3 -1)                                    ('B', 'A')

Seed strand set: ('A', 'B')
Ordered Wirtinger number: 2
